# Laboratorio 00 — Verificación del entorno
**Curso:** Deep Learning y Agentes Inteligentes
**Docente:** Ing. Julián René Muñoz Burbano
**Proyecto:** DriveGuard Lab — Detección de somnolencia y riesgo del conductor

**Objetivo:** comprobar que el entorno de Anaconda está listo antes de la clase: librerías, carpetas,
modelo FaceLandmarker, sonidos de alarma, MongoDB y cámara.

Ejecute las celdas en orden. Cada celda imprime `[OK]` o explica qué falta.

## 1. Ruta del proyecto
Agregamos la raíz del proyecto a `sys.path` para poder importar `core/` desde el notebook.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Directorio del proyecto:', ROOT)
print('Python:', sys.executable)

## 2. Librerías instaladas

In [ ]:
import importlib
paquetes = ['mediapipe', 'cv2', 'numpy', 'pandas', 'matplotlib', 'plotly', 'torch', 'transformers',
            'fastapi', 'pymongo', 'streamlit', 'streamlit_webrtc', 'av', 'pygame', 'sklearn', 'dotenv']
faltantes = []
for nombre in paquetes:
    try:
        mod = importlib.import_module(nombre)
        print(f'[OK] {nombre:18s} {getattr(mod, "__version__", "")}')
    except Exception as exc:
        faltantes.append(nombre)
        print(f'[FALTA] {nombre:14s} -> {type(exc).__name__}: {exc}')
print('\nTodo instalado.' if not faltantes else f'\nInstale con: python -m pip install -r requirements.txt')

## 3. Carpetas de trabajo
En el laboratorio 02 de SentimentLab V3 la celda de cámara fallaba porque la carpeta `data/` no venía en el zip.
En DriveGuard Lab, `core/config.py` crea automáticamente `data/`, `data/capturas`, `data/videos`, `data/ejemplos`,
`data/ojos`, `models/` y `assets/` al importarse.

In [ ]:
from core.config import asegurar_carpetas, DATA_DIR, CAPTURAS_DIR, VIDEOS_DIR, MODELS_DIR, ASSETS_DIR
asegurar_carpetas()
for carpeta in (DATA_DIR, CAPTURAS_DIR, VIDEOS_DIR, MODELS_DIR, ASSETS_DIR):
    print('[OK]' if carpeta.exists() else '[ERROR]', carpeta)

## 4. Modelo MediaPipe FaceLandmarker
Modelo preentrenado de Google (~3,6 MB) que entrega 478 puntos del rostro. Requiere internet la primera vez.

In [ ]:
from core.config import LANDMARKER_PATH, LANDMARKER_URL
from core.landmarks import descargar_modelo
try:
    ruta = descargar_modelo()
    print(f'[OK] {ruta} ({ruta.stat().st_size/1e6:.1f} MB)')
except Exception as exc:
    print('[ERROR] No se pudo descargar:', exc)
    print('Descárguelo a mano desde', LANDMARKER_URL, 'y guárdelo en', LANDMARKER_PATH)

## 5. Sonidos de alarma y archivos de ejemplo

In [ ]:
from core.alarma import generar_sonidos
from core.impacto import generar_csv_ejemplo
from core.config import EJEMPLOS_DIR
print(generar_sonidos())
print(generar_csv_ejemplo(EJEMPLOS_DIR / 'acelerometro_choque.csv', choque=True))
print(generar_csv_ejemplo(EJEMPLOS_DIR / 'acelerometro_sin_choque.csv', choque=False))

## 6. Prueba de sonido
Cambie `SONAR = True` para escuchar la alarma por los parlantes del equipo.

In [ ]:
SONAR = False
from core.alarma import crear_reproductor
rep = crear_reproductor()
print('Reproductor disponible:', rep.nombre)
if SONAR:
    import time
    rep.reproducir('media', bucle=False); time.sleep(1.2)
    rep.reproducir('alta', bucle=False); time.sleep(1.5)
else:
    print('Prueba de sonido desactivada')

## 7. MongoDB
Debe estar instalado y en ejecución (servicio *MongoDB Server* en Windows) o configurado en Atlas (`MONGO_URI` en `.env`).

In [ ]:
from core import storage
from core.config import MONGO_URI, MONGO_DB
print('URI:', MONGO_URI, '| Base:', MONGO_DB)
print('[OK] MongoDB responde' if storage.ping() else '[AVISO] MongoDB no responde: la app funciona, pero no guardará sesiones')

## 8. Cámara local (opcional)
Accede a la webcam del **equipo donde corre el kernel**. Cambie `USE_CAMERA = True` para probar.

In [ ]:
USE_CAMERA = False
import cv2
from core.config import CAPTURAS_DIR
if USE_CAMERA:
    cap = cv2.VideoCapture(0)
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise RuntimeError('No se pudo abrir la cámara (¿otra aplicación la está usando?)')
    destino = CAPTURAS_DIR / 'captura_prueba.jpg'
    cv2.imwrite(str(destino), frame)
    print('[OK] Captura guardada en', destino, frame.shape)
else:
    print('Captura desactivada: cambie USE_CAMERA a True para probar')

## 9. Siguientes pasos
1. Iniciar la API: `python -m uvicorn backend.main:app --reload --port 8000`
2. Iniciar la interfaz: `python -m streamlit run frontend/app.py`
3. Continuar con `03_Somnolencia_EAR_MAR.ipynb`